# Build an MCP Server: Python + TypeScript

**Phase 03** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-03/07-build-mcp-server.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q mcp

import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    pass  # Running locally — set OPENAI_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("OPENAI_API_KEY")))

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
L07: Build an MCP Server - Python
appliedaifromscratch.com | Phase 03

A complete MCP server for a product catalog database.
Exposes: search_products tool, get_product tool,
         product://catalog/{category} resource, product_summary prompt.

Run:
    pip install mcp
    python main.py                    # runs as stdio server
    mcp dev main.py                   # runs in the mcp inspector

Claude Desktop config (add to claude_desktop_config.json):
    {
      "mcpServers": {
        "product-catalog": {
          "command": "python",
          "args": ["/absolute/path/to/main.py"]
        }
      }
    }
"""

import json
import sqlite3
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("product-catalog")

### Database

In [ ]:
_db: sqlite3.Connection | None = None


def get_db() -> sqlite3.Connection:
    """Create and seed an in-memory SQLite database."""
    conn = sqlite3.connect(":memory:")
    conn.row_factory = sqlite3.Row
    conn.execute("""
        CREATE TABLE IF NOT EXISTS products (
            id          TEXT    PRIMARY KEY,
            name        TEXT    NOT NULL,
            category    TEXT    NOT NULL,
            price_cents INTEGER NOT NULL,
            description TEXT,
            in_stock    INTEGER DEFAULT 1
        )
    """)
    products = [
        ("P001", "Mechanical Keyboard",   "electronics", 12999, "TKL, Cherry MX Brown switches", 1),
        ("P002", "USB-C Hub 7-port",       "electronics",  4999, "4K HDMI, 100W PD, SD card reader", 1),
        ("P003", "Standing Desk Mat",      "office",        3499, "Anti-fatigue foam, 36x24 inches", 1),
        ("P004", "Monitor Arm",            "office",        8999, "Gas spring, holds up to 32-inch dual monitors", 1),
        ("P005", "Wireless Headphones",    "electronics", 19999, "Active noise cancellation, 30hr battery", 0),
        ("P006", "Laptop Stand",           "office",        5999, "Aluminum, 6 adjustable height positions", 1),
        ("P007", "Webcam 4K",              "electronics", 14999, "Auto-focus, excellent low-light performance", 1),
        ("P008", "Cable Management Kit",   "office",        1299, "Velcro straps, adhesive clips, under-desk tray", 1),
        ("P009", "Desk Lamp LED",          "office",        4499, "3 color temps, USB-A charging port", 1),
        ("P010", "Wrist Rest Set",         "office",        2999, "Memory foam for keyboard and mouse", 1),
    ]
    conn.executemany(
        "INSERT OR IGNORE INTO products VALUES (?,?,?,?,?,?)", products
    )
    conn.commit()
    return conn


def db() -> sqlite3.Connection:
    global _db
    if _db is None:
        _db = get_db()
    return _db

### Tool: search_products

In [ ]:
@mcp.tool()
def search_products(query: str, limit: int = 10) -> list[dict]:
    """
    Search products by name or description.

    Returns in-stock products only. Uses parameterized queries to prevent
    SQL injection. Limit is clamped to 1-50.

    Args:
        query: Search term to match against product name or description
        limit: Maximum number of results to return (1-50, default 10)
    """
    if not query or not query.strip():
        return []

    safe_limit = max(1, min(50, limit))
    search_term = f"%{query.strip()}%"

    cursor = db().execute(
        """
        SELECT id, name, category, price_cents, description, in_stock
        FROM products
        WHERE (name LIKE ? OR description LIKE ?)
          AND in_stock = 1
        ORDER BY name
        LIMIT ?
        """,
        (search_term, search_term, safe_limit),
    )
    return [dict(row) for row in cursor.fetchall()]

### Tool: get_product

In [ ]:
@mcp.tool()
def get_product(product_id: str) -> dict | None:
    """
    Get a single product by its ID.

    Returns the full product record, or null if not found.

    Args:
        product_id: The product ID (e.g. 'P001')
    """
    cursor = db().execute(
        "SELECT id, name, category, price_cents, description, in_stock "
        "FROM products WHERE id = ?",
        (product_id,),
    )
    row = cursor.fetchone()
    return dict(row) if row else None

### Resource: product://catalog/{category}

In [ ]:
@mcp.resource("product://catalog/{category}")
def get_catalog_by_category(category: str) -> str:
    """
    Return in-stock products in a category as JSON.

    URI examples:
        product://catalog/electronics   - all electronics
        product://catalog/office        - all office products
        product://catalog/all           - full catalog

    Args:
        category: Product category, or 'all' for the full catalog
    """
    if category == "all":
        cursor = db().execute(
            "SELECT id, name, category, price_cents, description "
            "FROM products WHERE in_stock = 1 ORDER BY category, name"
        )
    else:
        cursor = db().execute(
            "SELECT id, name, category, price_cents, description "
            "FROM products WHERE category = ? AND in_stock = 1 ORDER BY name",
            (category,),
        )
    rows = [dict(r) for r in cursor.fetchall()]
    return json.dumps(rows, indent=2)

### Prompt: product_summary

In [ ]:
@mcp.prompt()
def product_summary(product_id: str, style: str = "brief") -> str:
    """
    Generate a prompt to write a product summary.

    Args:
        product_id: The product ID to summarize (e.g. 'P001')
        style: 'brief' for a 1-2 sentence snippet, 'detailed' for a full description
    """
    product = get_product(product_id)
    if not product:
        return f"No product found with ID '{product_id}'. Please check the product ID."

    price = f"${product['price_cents'] / 100:.2f}"
    stock_status = "In stock" if product["in_stock"] else "Out of stock"

    base = (
        f"Product: {product['name']} (ID: {product['id']})\n"
        f"Category: {product['category']}\n"
        f"Price: {price}\n"
        f"Status: {stock_status}\n"
        f"Description: {product['description']}\n\n"
    )

    if style == "brief":
        instruction = (
            "Write a 1-2 sentence product summary suitable for a search result snippet. "
            "Lead with the most distinctive feature."
        )
    else:
        instruction = (
            "Write a detailed product description with:\n"
            "1. A compelling headline\n"
            "2. Key features as bullet points\n"
            "3. Ideal use case (who is this for?)\n"
            "4. A closing sentence on value for money"
        )

    return base + instruction


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

### Demo

In [ ]:
# Run as stdio server (for Claude Desktop or mcp dev)
# Logs go to stderr; stdout is reserved for JSON-RPC messages
import sys
print("Starting product-catalog MCP server on stdio...", file=sys.stderr)
mcp.run(transport="stdio")

---

## Self-Check

Answer these without running code first:

**1. What is the security risk, and how should it be fixed?**

- A. No risk; LIKE queries are read-only so injection cannot modify data
- B. SQL injection: a crafted query string can bypass the LIKE clause and return arbitrary data or execute additional statements. Fix: use parameterized queries with ? placeholders
- C. Performance risk only; the fix is to add an index on the name column
- D. The query will fail because f-strings cannot be used inside execute() calls

**2. Which MCP primitive fits best?**

- A. Tool, because it requires an ID parameter
- B. Resource, because the report is stable, addressable by ID, and read-only
- C. Prompt, because the report content is used as context for the LLM
- D. Sampling, because generating the report requires LLM inference

**3. What is most likely causing the connection failure?**

- A. claude_desktop_config.json has the wrong command path
- B. print() statements write to stdout, which Claude Desktop reads as JSON-RPC messages, causing parse failures before the handshake begins
- C. Claude Desktop does not support Python servers, only TypeScript
- D. The server must use HTTP transport, not stdio, for Claude Desktop

**4. What is the key operational advantage of option A?**

- A. MCP servers are faster than direct database connections
- B. A schema change or security fix in the database query logic only needs to be made in one place
- C. MCP servers automatically scale to handle more requests
- D. Option B is always preferable because direct connections have less latency

**5. What are two concrete problems with this approach?**

- A. JSON-RPC cannot forward HTTP methods; the tool will fail at runtime
- B. The LLM must guess endpoint paths and parameter shapes rather than using typed schemas, and the host cannot distinguish safe read operations from mutations
- C. A single tool can handle at most 10 parameters; 40 endpoints will exceed the limit
- D. FastMCP does not support tools with 'method' as a parameter name

**6. Which claude_desktop_config.json entry is correct?**

- A. {"command": "server.py", "args": []}
- B. {"command": "python3", "args": ["/home/alice/projects/catalog/server.py"]}
- C. {"command": "mcp", "args": ["run", "server.py"]}
- D. {"command": "curl", "args": ["http://localhost:8000/mcp"]}

**7. What is the correct way to add debug logging in a Node.js MCP server?**

- A. console.log() is fine; only output before server.connect() breaks stdio
- B. Use process.stderr.write() or console.error() instead of console.log(), because console.log writes to stdout which is reserved for JSON-RPC
- C. Add a logging: true flag to the McpServer constructor to enable built-in logging
- D. Use a file logger that writes to /tmp/mcp-debug.log

**8. What should the prompt handler return when the product is not found?**

- A. Raise a KeyError so the MCP framework returns a protocol error to the host
- B. Return a messages array with a user message explaining the product was not found, so the LLM can inform the user gracefully
- C. Return an empty messages array so the LLM skips the prompt
- D. Return None; the FastMCP framework will handle missing returns automatically

_Answers are in `checks.json` in the lesson directory._